# 01 - EDA and Preprocessing

**Project:** Predictive Maintenance for Transit Systems

**Dataset:** Synthetic transit asset inventory, failure history, sensor telemetry, and maintenance logs based on FTA NTD and WMATA operational patterns.

**Objective:** Explore the data, understand failure patterns, and engineer features for survival analysis and anomaly detection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Libraries loaded.")

## Load Datasets

In [ ]:
DATA_DIR = '../data'

assets = pd.read_csv(f'{DATA_DIR}/transit_assets.csv')
assets['installation_date'] = pd.to_datetime(assets['installation_date'])
assets['last_maintenance'] = pd.to_datetime(assets['last_maintenance'])

failures = pd.read_csv(f'{DATA_DIR}/failure_history.csv')
failures['failure_date'] = pd.to_datetime(failures['failure_date'])

telemetry = pd.read_csv(f'{DATA_DIR}/sensor_telemetry.csv')
telemetry['date'] = pd.to_datetime(telemetry['date'])

maintenance = pd.read_csv(f'{DATA_DIR}/maintenance_logs.csv')
maintenance['maintenance_date'] = pd.to_datetime(maintenance['maintenance_date'])

print(f"Assets: {len(assets):,} records")
print(f"Failures: {len(failures):,} records")
print(f"Telemetry: {len(telemetry):,} records")
print(f"Maintenance: {len(maintenance):,} records")

## Asset Inventory Overview

In [ ]:
print("=== ASSET INVENTORY ===")
print(f"Asset types: {assets['asset_type'].nunique()}")
print(f"Lines covered: {assets['line'].nunique()}")
print(f"\nAsset type distribution:")
print(assets['asset_type'].value_counts())

print(f"\nCriticality distribution:")
print(assets['criticality'].value_counts(normalize=True).round(3))

print(f"\nAge statistics:")
print(assets['age_years'].describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Asset types
assets['asset_type'].value_counts().plot(kind='bar', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Asset Type Distribution')
axes[0,0].set_xlabel('Asset Type')
axes[0,0].tick_params(axis='x', rotation=45)

# Age distribution
assets['age_years'].hist(bins=30, ax=axes[0,1], color='coral', edgecolor='black')
axes[0,1].set_title('Asset Age Distribution (Years)')
axes[0,1].set_xlabel('Age (years)')

# Criticality
assets['criticality'].value_counts().plot(kind='bar', ax=axes[1,0], color='seagreen')
axes[1,0].set_title('Criticality Distribution')
axes[1,0].tick_params(axis='x', rotation=45)

# Condition vs Age
axes[1,1].scatter(assets['age_years'], assets['condition_score'], alpha=0.5, color='purple')
axes[1,1].set_title('Condition Score vs Age')
axes[1,1].set_xlabel('Age (years)')
axes[1,1].set_ylabel('Condition Score (1-10)')

plt.tight_layout()
plt.savefig('../figures/01_asset_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../figures/01_asset_overview.png")

## Failure History Analysis

In [ ]:
print("=== FAILURE HISTORY ===")
print(f"Total failures: {len(failures):,}")
print(f"Unique assets with failures: {failures['asset_id'].nunique()}")
print(f"Failure rate: {failures['asset_id'].nunique() / len(assets):.1%}")

print(f"\nFailure type distribution:")
print(failures['failure_type'].value_counts(normalize=True).round(3))

print(f"\nSeverity distribution:")
print(failures['severity'].value_counts(normalize=True).round(3))

print(f"\nDowntime statistics (hours):")
print(failures['downtime_hours'].describe().round(2))

print(f"\nRepair cost statistics ($):")
print(failures['repair_cost'].describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Failures over time
failures['failure_date'].dt.to_period('M').value_counts().sort_index().plot(ax=axes[0,0], color='crimson')
axes[0,0].set_title('Failures Over Time (Monthly)')
axes[0,0].set_xlabel('Month')

# Failure type
failures['failure_type'].value_counts().plot(kind='bar', ax=axes[0,1], color='teal')
axes[0,1].set_title('Failure Type Distribution')
axes[0,1].tick_params(axis='x', rotation=45)

# Downtime distribution
failures['downtime_hours'].hist(bins=30, ax=axes[1,0], color='orange', edgecolor='black')
axes[1,0].set_title('Downtime Distribution (Hours)')
axes[1,0].set_xlabel('Downtime (hours)')

# Repair cost by severity
sns.boxplot(data=failures, x='severity', y='repair_cost', ax=axes[1,1], palette='YlOrRd')
axes[1,1].set_title('Repair Cost by Severity')
axes[1,1].set_ylabel('Repair Cost ($)')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../figures/01_failure_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../figures/01_failure_analysis.png")

## Sensor Telemetry Exploration

In [ ]:
print("=== SENSOR TELEMETRY ===")
print(f"Total readings: {len(telemetry):,}")
print(f"Sensor types: {telemetry['sensor_type'].unique()}")
print(f"Date range: {telemetry['date'].min()} to {telemetry['date'].max()}")

print(f"\nReading statistics by sensor:")
print(telemetry.groupby('sensor_type')['reading'].describe().round(3))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, sensor in enumerate(telemetry['sensor_type'].unique()):
    data = telemetry[telemetry['sensor_type'] == sensor]
    ax = axes[idx // 2, idx % 2]
    ax.hist(data['reading'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_title(f'{sensor.capitalize()} Distribution')
    ax.set_xlabel(f'{sensor.capitalize()} ({data["unit"].iloc[0]})')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig('../figures/01_sensor_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../figures/01_sensor_distributions.png")

## Feature Engineering

In [ ]:
# Merge asset and failure data for survival analysis features
asset_failures = assets.merge(
    failures.groupby('asset_id').agg(
        failure_count=('failure_date', 'count'),
        first_failure=('failure_date', 'min'),
        last_failure=('failure_date', 'max'),
        total_downtime=('downtime_hours', 'sum'),
        total_repair_cost=('repair_cost', 'sum'),
        avg_severity=('severity', lambda x: x.map({'Minor':1, 'Moderate':2, 'Major':3, 'Severe':4}).mean())
    ).reset_index(),
    on='asset_id', how='left'
)

# Fill assets with no failures
asset_failures['failure_count'] = asset_failures['failure_count'].fillna(0)
asset_failures['total_downtime'] = asset_failures['total_downtime'].fillna(0)
asset_failures['total_repair_cost'] = asset_failures['total_repair_cost'].fillna(0)

# Days since last maintenance
asset_failures['days_since_maintenance'] = (datetime(2024, 12, 31) - asset_failures['last_maintenance']).dt.days

# Maintenance frequency
maintenance_freq = maintenance.groupby('asset_id').size().reset_index(name='maintenance_frequency')
asset_failures = asset_failures.merge(maintenance_freq, on='asset_id', how='left')
asset_failures['maintenance_frequency'] = asset_failures['maintenance_frequency'].fillna(0)

# Encode categorical features
asset_failures['criticality_encoded'] = asset_failures['criticality'].map(
    {'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 4}
)

# Survival target: time to first failure (censored if no failure)
asset_failures['time_to_failure'] = np.where(
    pd.notna(asset_failures['first_failure']),
    (asset_failures['first_failure'] - asset_failures['installation_date']).dt.days,
    (datetime(2024, 12, 31) - asset_failures['installation_date']).dt.days
)
asset_failures['failed'] = pd.notna(asset_failures['first_failure']).astype(int)

print("Feature engineering complete.")
print(f"\nFeature summary:")
feature_cols = ['age_years', 'condition_score', 'daily_usage_hours', 'days_since_maintenance',
                'maintenance_frequency', 'criticality_encoded', 'time_to_failure', 'failed']
print(asset_failures[feature_cols].describe().round(2))

In [ ]:
# Save processed features
asset_failures.to_csv(f'{DATA_DIR}/processed_features.csv', index=False)
print(f"Saved processed features: {DATA_DIR}/processed_features.csv")

# Summary stats
print(f"\n=== DATA QUALITY ===")
print(f"Assets tracked: {len(asset_failures)}")
print(f"Assets with failures: {asset_failures['failed'].sum()}")
print(f"Censored assets (no failure): {(asset_failures['failed'] == 0).sum()}")
print(f"Median time to failure: {asset_failures[asset_failures['failed']==1]['time_to_failure'].median():.0f} days")
print(f"Mean time to failure: {asset_failures[asset_failures['failed']==1]['time_to_failure'].mean():.0f} days")

## Correlation Analysis

In [ ]:
corr_cols = ['age_years', 'condition_score', 'daily_usage_hours', 
             'days_since_maintenance', 'maintenance_frequency', 
             'criticality_encoded', 'failure_count', 'time_to_failure']

corr = asset_failures[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f',
          square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../figures/01_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../figures/01_correlation_matrix.png")

## Key Findings

1. **Failure Rate:** ~{:.0%} of assets experienced at least one failure
2. **Age Effect:** Older assets show higher failure rates and lower condition scores
3. **Criticality:** High/Critical assets account for disproportionate repair costs
4. **Sensor Patterns:** Temperature and vibration show clear degradation signals
5. **Maintenance Gap:** Assets with longer gaps since maintenance have higher failure risk

Next: Survival Analysis (Notebook 02)